<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 02. SimCLR: augmentation과 NT-Xent

## A Simple Framework for Contrastive Learning of Visual Representations

- **원 논문:** [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709)
- **저자 / 발표:** Ting Chen, Simon Kornblith, Mohammad Norouzi, Geoffrey Hinton · ICML (2020)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** cosine similarity, cross entropy, augmentation
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 두 augmentation view, encoder f, projection g, 대칭 NT-Xent를 구현합니다.

**축소하거나 생략한 부분:** ImageNet·ResNet-50·batch 4096 대신 96개 8×8 이미지와 작은 MLP를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 2 | x→두 view→f→g | view 비동일성·shape |
| §2.1, Eq. (1) | L2-normalized cosine | 대각/범위 |
| §2.1, Eq. (2) | 2N 대칭 NT-Xent | self mask·positive index |
| Algorithm 1 및 §2.3 | projection 학습·h 평가 | same/different class cosine |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\ell_{i,j}=-\log\frac{\exp(\operatorname{sim}(z_i,z_j)/\tau)}{\sum_{k\ne i}\exp(
\operatorname{sim}(z_i,z_k)/\tau)}$$

- **기호 정의:** i와 j는 같은 샘플의 두 view, z는 projection, τ는 온도입니다.
- **수식의 역할:** 강한 augmentation pair를 positive로 두고 2B-2개 view를 negative로 사용합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 Figure 2을 구현합니다.
- **입출력 shape:** 두 view [B, 1, 8, 8] → projection [2B, D] → similarity [2B, 2B]
- **평가:** 대칭 NT-Xent와 같은 클래스/다른 클래스 cosine gap
- **원문 대비 한계:** ImageNet·ResNet-50·batch 4096 대신 96개 8×8 이미지와 작은 MLP를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** Figure 2
- **이 셀의 책임:** x→두 view→f→g
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 비동일성·shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def make_views(x: Tensor):
    def aug():
        scale = 0.75 + 0.5 * torch.rand(len(x), 1, 1, 1, device=x.device)
        return (x * scale + 0.04 * torch.randn_like(x)).clamp(0, 1)

    return aug(), aug()


a, b = make_views(images[:96])
assert a.shape == b.shape == (96, 1, 8, 8) and not torch.equal(a, b)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1) / §2.1, Eq. (2)
- **이 셀의 책임:** L2-normalized cosine / 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 대각/범위 / self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def nt_xent(z1: Tensor, z2: Tensor, temperature=0.2):
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    n = len(z1)
    logits = z @ z.T / temperature
    logits.fill_diagonal_(-1e9)
    targets = (torch.arange(2 * n, device=logits.device) + n) % (2 * n)
    return F.cross_entropy(logits, targets), logits


probe, _ = nt_xent(torch.randn(8, 6), torch.randn(8, 6))
assert torch.isfinite(probe)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 2 / §2.1, Eq. (1) / §2.1, Eq. (2) / Algorithm 1 및 §2.3
- **이 셀의 책임:** x→두 view→f→g / L2-normalized cosine / 2N 대칭 NT-Xent / projection 학습·h 평가
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** view 비동일성·shape / 대각/범위 / self mask·positive index / same/different class cosine. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
encoder = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16))
projector = nn.Sequential(nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 8))
encoder, projector = ACCELERATOR.move(encoder, projector)
optimizer = torch.optim.Adam([*encoder.parameters(), *projector.parameters()], lr=0.02)
batch = ACCELERATOR.move(images[train_idx[:96]])
losses = []
for _ in range(110):
    a, b = make_views(batch)
    loss, _ = nt_xent(projector(encoder(a)), projector(encoder(b)))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    h = F.normalize(encoder(ACCELERATOR.move(images[test_idx])), dim=1).cpu()
    sim = h @ h.T
    y = class_ids[test_idx]
    eye = torch.eye(len(y), dtype=torch.bool)
    same = (y[:, None] == y[None, :]) & ~eye
    diff = y[:, None] != y[None, :]
    sm = float(sim[same].mean())
    dm = float(sim[diff].mean())
assert losses[-1] < losses[0] and sm > dm
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("NT-Xent")
ax[1].bar(["same", "different"], [sm, dm])
ax[1].set_title("encoder h cosine")
plt.tight_layout()
plt.show()

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 2 / §2.1, Eq. (1) / §2.1, Eq. (2)
- **이 셀의 책임:** x→두 view→f→g / L2-normalized cosine / 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 비동일성·shape / 대각/범위 / self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    temperature: float = 0.2
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    first = (batch + 0.04 * torch.randn_like(batch)).clamp(0.0, 1.0)
    second = torch.flip(batch, dims=[3])
    second = (second + 0.04 * torch.randn_like(second)).clamp(0.0, 1.0)
    return first, second


class SimclrPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
        )
        self.projector = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )

    def forward(self, view: Tensor) -> Tensor:
        return F.normalize(self.projector(self.encoder(view)), dim=1)

    def nt_xent_logits(
        self,
        first_projection: Tensor,
        second_projection: Tensor,
    ) -> tuple[Tensor, Tensor]:
        batch_size = len(first_projection)
        representations = torch.cat(
            [first_projection, second_projection],
            dim=0,
        )
        pair_count = 2 * batch_size
        logits = representations @ representations.T
        logits = logits / self.config.temperature
        self_mask = torch.eye(
            pair_count,
            dtype=torch.bool,
            device=logits.device,
        )
        logits = logits.masked_fill(self_mask, -torch.inf)
        anchors = torch.arange(pair_count, device=logits.device)
        positive_index = (anchors + batch_size) % pair_count

        expected_positive = torch.cat(
            [anchors[batch_size:], anchors[:batch_size]],
        )
        assert logits.shape == (pair_count, pair_count)
        assert torch.isneginf(logits.diagonal()).all()
        assert torch.equal(positive_index, expected_positive)
        return logits, positive_index

    def compute_loss(self, first: Tensor, second: Tensor) -> Tensor:
        first_projection = self(first)
        second_projection = self(second)
        logits, positive_index = self.nt_xent_logits(
            first_projection,
            second_projection,
        )
        return F.cross_entropy(logits, positive_index)

    def training_step(self, batch: Tensor) -> Tensor:
        first, second = prepare_views(batch)
        return self.compute_loss(first, second)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.1, Eq. (2)
- **이 셀의 책임:** 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: SimclrPortfolioModel, batch: Tensor) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2.1, Eq. (2)
- **이 셀의 책임:** 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: SimclrPortfolioModel,
    batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        first, second = prepare_views(batch)
        logits, positive_index = model.nt_xent_logits(
            model(first),
            model(second),
        )
        accuracy = (logits.argmax(1) == positive_index).float().mean()
        loss = F.cross_entropy(logits, positive_index)
    return {
        "nt_xent_pair_top1": float(accuracy.cpu()),
        "nt_xent_loss": float(loss.cpu()),
    }


portfolio_train_index = train_idx[:64]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_test_images = ACCELERATOR.move(
    images[portfolio_train_index],
    images[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(SimclrPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_train_images)
portfolio_metrics = evaluate_portfolio_model(portfolio_model, portfolio_test_images)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet·ResNet-50·batch 4096 대신 96개 8×8 이미지와 작은 MLP를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.